<p style="text-align:right"><b><span style="color:#2E7D32">Pública</span></b></p>

# Reto | Generación de un modelo inteligente de clasificación con redes neuronales profundas para la detección de amenazas aéreas

**Alumno:** _[Escribe aquí tu nombre completo]_
**Curso:** Ciencia de Datos — Módulo 7, Reto (DS_C7_SC3)

## Objetivo
Desarrollar y comparar tres modelos de clasificación de imágenes capaces de identificar **6 tipos de objetos aéreos** (avión, cohete, dron, helicóptero, jet y misil) para integrarlos al sistema de seguridad perimetral de la base que desarrolla la vacuna contra la plaga zombi:

1. **Modelo 1:** red neuronal convolucional (CNN) diseñada **desde cero**, con búsqueda de hiperparámetros y aumento de datos.
2. **Modelo 2:** CNN basada en un **modelo pre-entrenado** (transferencia de aprendizaje con MobileNetV2).
3. **Modelo 3:** modelo generado con **Teachable Machine** usando solo el conjunto de entrenamiento.

Los tres modelos se exportan, se vuelven a importar y se comparan con el **mismo conjunto de prueba**, usando la **exactitud (accuracy)** como indicador principal para seleccionar el mejor.

## Contenido
1. Configuración del entorno
2. Exploración de la base de datos
3. Preprocesamiento y limpieza
4. Conjuntos de entrenamiento y prueba
5. Canalización de datos y aumento de datos
6. Modelo 1: CNN desde cero
7. Modelo 2: modelo pre-entrenado (MobileNetV2)
8. Modelo 3: Teachable Machine
9. Importación y comparación de los tres modelos
10. Selección del mejor modelo
11. Exportación de los modelos
12. Conclusiones

> **Antes de ejecutar:** guarda este cuaderno en la misma carpeta que contiene la carpeta `datos/objetos` (si no existe, el cuaderno la descarga de GitHub; en Colab conviene activar la GPU T4) y ejecuta las celdas en orden (*Ejecutar todo*). Deja visibles todas las salidas antes de entregar.

## 1. Configuración del entorno
Instalamos TensorFlow (si no está instalado), importamos las bibliotecas, fijamos una semilla para que los resultados sean reproducibles y verificamos si hay GPU disponible.

In [ ]:
%pip install -q tensorflow tf_keras seaborn scikit-learn tqdm pillow

In [ ]:
import os, sys, random, hashlib, shutil, zipfile, json, time, unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
import keras
from keras import layers

SEMILLA = 42
keras.utils.set_random_seed(SEMILLA)   # fija semillas de Python, NumPy y TensorFlow

print('TensorFlow:', tf.__version__)
print('Keras     :', keras.__version__)
print('GPU       :', tf.config.list_physical_devices('GPU') or 'No disponible: se entrenará con CPU (más lento)')

TensorFlow: 2.21.0
Keras     : 3.13.2
GPU       : No disponible: se entrenará con CPU (más lento)


### 1.1 Rutas de trabajo
La base de datos se descargó y se descomprimió en la carpeta del proyecto (pasos 1 y 2 de las instrucciones). Los datos se cargan directamente desde esa carpeta local. Si la carpeta `datos/objetos` no existe (por ejemplo, al abrir el cuaderno en Google Colab), se descarga automáticamente desde el repositorio de GitHub. Ajusta `RUTA_BASE` si tus datos están en otra ubicación.

- **Origen:** carpeta `datos/objetos`, con una subcarpeta por clase.
- **Datos preparados:** imágenes estandarizadas en `datos_preparados/entrenamiento` y `datos_preparados/prueba`.
- **Modelos y resultados:** se guardan en la carpeta `resultados`.

In [ ]:
RUTA_BASE     = Path.cwd()                         # carpeta donde está este cuaderno
RUTA_ORIGEN   = RUTA_BASE / 'datos' / 'objetos'    # <- ajusta si es necesario
RUTA_LOCAL    = RUTA_BASE / 'datos_preparados'
RUTA_PROYECTO = RUTA_BASE / 'resultados'
RUTA_MODELOS  = RUTA_PROYECTO / 'modelos'
RUTA_TM       = RUTA_MODELOS / 'teachable_machine'
RUTA_TRAIN    = RUTA_LOCAL / 'entrenamiento'
RUTA_TEST     = RUTA_LOCAL / 'prueba'

for ruta in (RUTA_PROYECTO, RUTA_MODELOS, RUTA_TM):
    ruta.mkdir(parents=True, exist_ok=True)

TAM_IMG = (224, 224)   # tamaño estándar para los tres modelos (Teachable Machine también usa 224 x 224)
LOTE    = 32

# Si la base de datos no está en la carpeta (por ejemplo, en Colab), se descarga desde GitHub
URL_DATOS = 'https://github.com/obedgar90/reto-amenazas-aereas/releases/download/datos-v1/datos.zip'
if not RUTA_ORIGEN.exists():
    import urllib.request
    print('Descargando la base de datos desde GitHub (aprox. 1.6 GB)...')
    zip_datos = RUTA_BASE / 'datos.zip'
    urllib.request.urlretrieve(URL_DATOS, zip_datos)
    with zipfile.ZipFile(zip_datos) as z:
        z.extractall(RUTA_BASE)
    zip_datos.unlink()
    print('Base de datos descargada y descomprimida.')

CLASES = sorted(d.name for d in RUTA_ORIGEN.iterdir() if d.is_dir())
NUM_CLASES = len(CLASES)
print('Origen :', RUTA_ORIGEN)
print('Clases :', CLASES)
assert NUM_CLASES == 6, 'Se esperaban 6 clases; revisa RUTA_ORIGEN.'

FileNotFoundError: [Errno 2] No such file or directory: '/content/datos/objetos'

## 2. Exploración de la base de datos
Recorremos todas las imágenes para obtener un inventario con: clase, extensión del archivo, formato real (detectado con Pillow), modo de color, dimensiones y una huella **MD5** del contenido, que usaremos para detectar duplicados. Este paso puede tardar algunos minutos porque lee las 8,530 imágenes.

In [ ]:
registros = []
for clase in CLASES:
    for archivo in tqdm(sorted((RUTA_ORIGEN / clase).iterdir()), desc=clase):
        if not archivo.is_file():
            continue
        datos = archivo.read_bytes()
        fila = {'clase': clase, 'ruta': str(archivo), 'extension': archivo.suffix.lower() or '(sin extensión)',
                'md5': hashlib.md5(datos).hexdigest(), 'legible': True,
                'formato': None, 'modo': None, 'ancho': None, 'alto': None}
        try:
            with Image.open(archivo) as img:
                img.verify()                       # detecta archivos dañados
            with Image.open(archivo) as img:
                fila.update(formato=img.format, modo=img.mode, ancho=img.width, alto=img.height)
        except Exception:
            fila['legible'] = False
        registros.append(fila)

inventario = pd.DataFrame(registros)
print(f'Total de imágenes: {len(inventario):,}')
print(f'Imágenes no legibles: {(~inventario.legible).sum()}')
inventario.groupby('clase').size().rename('imágenes').to_frame()

In [ ]:
print('Extensiones de archivo encontradas:')
display(inventario.extension.value_counts().rename('archivos').to_frame().T)
print('Formato real de la imagen (según su contenido):')
display(inventario.formato.value_counts(dropna=False).rename('archivos').to_frame().T)
print('Modo de color:')
display(inventario.modo.value_counts(dropna=False).rename('archivos').to_frame().T)
print('Dimensiones (píxeles):')
inventario[['ancho', 'alto']].describe().round(0)

**Observaciones:** las imágenes vienen con extensiones muy variadas (`.jpg`, `.png`, `.webp`, `.gif`, e incluso `.aspx`, `.php` o sin extensión válida), aunque su contenido real es JPEG, PNG, WEBP, GIF o MPO. También tienen modos de color distintos (RGB, paleta `P` y RGBA con transparencia) y dimensiones muy diferentes. Por eso es necesario estandarizarlas antes de entrenar.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 4))
conteo = inventario.clase.value_counts().reindex(CLASES)
sns.barplot(x=conteo.index, y=conteo.values, ax=ejes[0], color='steelblue')
ejes[0].set_title('Imágenes por clase (base original)')
ejes[0].set_ylabel('Imágenes')
for i, v in enumerate(conteo.values):
    ejes[0].text(i, v, f'{v:,}', ha='center', va='bottom')
ejes[1].scatter(inventario.ancho, inventario.alto, s=4, alpha=0.3)
ejes[1].set_xscale('log'); ejes[1].set_yscale('log')
ejes[1].set_title('Dimensiones de las imágenes'); ejes[1].set_xlabel('Ancho (px)'); ejes[1].set_ylabel('Alto (px)')
plt.tight_layout(); plt.show()

In [ ]:
def mostrar_cuadricula(rutas, titulos, columnas=6, tam=2.4, titulo_general=None):
    filas = int(np.ceil(len(rutas) / columnas))
    fig, ejes = plt.subplots(filas, columnas, figsize=(columnas * tam, filas * tam))
    for eje in np.ravel(ejes):
        eje.axis('off')
    for eje, ruta, titulo in zip(np.ravel(ejes), rutas, titulos):
        with Image.open(ruta) as img:
            eje.imshow(img.convert('RGB'))
        eje.set_title(titulo, fontsize=9)
    if titulo_general:
        fig.suptitle(titulo_general, fontsize=13)
    plt.tight_layout(); plt.show()

muestra = inventario[inventario.legible].groupby('clase').sample(6, random_state=SEMILLA)
mostrar_cuadricula(muestra.ruta.tolist(), muestra.clase.tolist(), titulo_general='Ejemplos de cada clase')

## 3. Preprocesamiento y limpieza
### 3.1 Detección de duplicados
Dos archivos con la misma huella MD5 son exactamente la misma imagen. Los duplicados causan dos problemas:

- **Fuga de información:** si una copia queda en entrenamiento y otra en prueba, la exactitud medida será artificialmente alta.
- **Etiquetas contradictorias:** si la misma imagen aparece en dos clases distintas, el modelo recibe señales opuestas.

In [ ]:
grupos = inventario[inventario.legible].groupby('md5')
resumen_dup = grupos.agg(copias=('clase', 'size'), clases=('clase', lambda c: ' + '.join(sorted(set(c)))))
resumen_dup['num_clases'] = resumen_dup.clases.str.count(r'\+') + 1

dup_misma = resumen_dup[(resumen_dup.copias > 1) & (resumen_dup.num_clases == 1)]
dup_cruz  = resumen_dup[resumen_dup.num_clases > 1]
print(f'Imágenes duplicadas dentro de la misma clase : {len(dup_misma):,} grupos ({(dup_misma.copias - 1).sum():,} copias sobrantes)')
print(f'Imágenes repetidas en clases distintas       : {len(dup_cruz):,} grupos ({dup_cruz.copias.sum():,} archivos)')
print('\nCombinaciones de clases con la misma imagen:')
dup_cruz.clases.value_counts().rename('grupos').to_frame()

In [ ]:
# Revisión visual: imágenes que aparecen a la vez en 'cohete' y en 'jet'
md5_cj = dup_cruz[dup_cruz.clases == 'cohete + jet'].index[:18]
ejemplos = inventario[inventario.md5.isin(md5_cj) & (inventario.clase == 'cohete')]
mostrar_cuadricula(ejemplos.ruta.tolist(), ['cohete + jet'] * len(ejemplos),
                   titulo_general='Imágenes repetidas en las carpetas "cohete" y "jet"')

**Decisión de limpieza:** la revisión visual muestra que la mayoría de las imágenes repetidas en `cohete` y `jet` son aviones de combate o aeronaves, no cohetes; es decir, son un error de etiquetado de la base. Como no es posible asignarles una etiqueta confiable sin revisar una por una, se aplican estas reglas:

1. Se eliminan las imágenes que no se pueden abrir (si las hay).
2. Se **eliminan por completo** los grupos de imágenes que aparecen en **más de una clase** (etiqueta ambigua).
3. De las imágenes repetidas dentro de una misma clase se conserva **una sola copia**.

La clase `cohete` queda con menos ejemplos; ese desbalance se compensa más adelante con **pesos por clase** durante el entrenamiento.

In [ ]:
md5_ambiguos = set(dup_cruz.index)
limpio = (inventario[inventario.legible & ~inventario.md5.isin(md5_ambiguos)]
          .drop_duplicates(subset='md5', keep='first')
          .reset_index(drop=True))

comparativo = pd.DataFrame({
    'original': inventario.clase.value_counts(),
    'después de limpieza': limpio.clase.value_counts(),
}).reindex(CLASES)
comparativo['eliminadas'] = comparativo['original'] - comparativo['después de limpieza']
comparativo.loc['TOTAL'] = comparativo.sum()
comparativo

### 3.2 Estandarización de las imágenes
Cada imagen se transforma de la misma manera:
- Se corrige la orientación según los metadatos EXIF.
- Las imágenes con transparencia (RGBA o paleta) se combinan sobre fondo blanco y todas se convierten a **RGB** (3 canales). De los GIF se toma el primer cuadro.
- Se redimensionan a **224 × 224 píxeles**.
- Se guardan como **JPEG**, con un nombre uniforme `clase_00000.jpg`.

In [ ]:
def preparar_imagen(ruta_origen, ruta_destino, tam=TAM_IMG):
    with Image.open(ruta_origen) as img:
        img.seek(0)                                  # primer cuadro (GIF / MPO)
        img = ImageOps.exif_transpose(img)
        if img.mode in ('RGBA', 'LA', 'P'):
            img = img.convert('RGBA')
            fondo = Image.new('RGBA', img.size, (255, 255, 255, 255))
            img = Image.alpha_composite(fondo, img)
        img = img.convert('RGB').resize(tam, Image.Resampling.LANCZOS)
        img.save(ruta_destino, 'JPEG', quality=95)

## 4. Conjuntos de entrenamiento y prueba
Se divide la base limpia en **80 % entrenamiento** y **20 % prueba**, de forma **estratificada** (cada clase conserva su proporción). El conjunto de prueba **no se usa** para entrenar ni ajustar ningún modelo; solo sirve para la comparación final. Del conjunto de entrenamiento se separará después un 20 % como **validación** para ajustar los hiperparámetros.

In [ ]:
entrenamiento_df, prueba_df = train_test_split(
    limpio, test_size=0.20, stratify=limpio.clase, random_state=SEMILLA)
entrenamiento_df = entrenamiento_df.assign(conjunto='entrenamiento')
prueba_df = prueba_df.assign(conjunto='prueba')

if RUTA_LOCAL.exists():
    shutil.rmtree(RUTA_LOCAL)

particion = []
for df, raiz in ((entrenamiento_df, RUTA_TRAIN), (prueba_df, RUTA_TEST)):
    for clase in CLASES:
        (raiz / clase).mkdir(parents=True, exist_ok=True)
    for i, fila in enumerate(tqdm(df.itertuples(), total=len(df), desc=raiz.name)):
        destino = raiz / fila.clase / f'{fila.clase}_{i:05d}.jpg'
        preparar_imagen(fila.ruta, destino)
        particion.append({'origen': fila.ruta, 'destino': str(destino), 'clase': fila.clase,
                          'conjunto': fila.conjunto, 'md5': fila.md5})

particion = pd.DataFrame(particion)
particion.to_csv(RUTA_PROYECTO / 'particion_entrenamiento_prueba.csv', index=False)  # trazabilidad
tabla = particion.pivot_table(index='clase', columns='conjunto', values='md5', aggfunc='count')
tabla['total'] = tabla.sum(axis=1)
tabla.loc['TOTAL'] = tabla.sum()
tabla

In [ ]:
# Comprobación: ninguna imagen de prueba está en entrenamiento
fuga = set(particion[particion.conjunto == 'prueba'].md5) & set(particion[particion.conjunto == 'entrenamiento'].md5)
print('Imágenes compartidas entre entrenamiento y prueba:', len(fuga))
assert not fuga

### 4.1 Paquete del conjunto de entrenamiento para Teachable Machine
El modelo 3 debe entrenarse **solamente con el conjunto de entrenamiento**. Se genera un archivo ZIP con una carpeta por clase y se guarda en la carpeta `resultados` para cargarlo en Teachable Machine (sección 8).

In [ ]:
zip_tm = shutil.make_archive(str(RUTA_PROYECTO / 'entrenamiento_teachable_machine'), 'zip', RUTA_TRAIN)
print('ZIP generado:', zip_tm, f'({os.path.getsize(zip_tm) / 1e6:.1f} MB)')
print('Carpeta para subir a Teachable Machine:', RUTA_TRAIN)

## 5. Canalización de datos y aumento de datos
Se cargan las imágenes con `image_dataset_from_directory`. Del conjunto de entrenamiento se reserva el 20 % para **validación**. Para acelerar el entrenamiento, los datos se guardan en memoria caché (como `uint8`) y se precargan en paralelo.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_raw, val_raw = keras.utils.image_dataset_from_directory(
    RUTA_TRAIN, validation_split=0.2, subset='both', seed=SEMILLA,
    image_size=TAM_IMG, batch_size=LOTE, class_names=CLASES)
test_raw = keras.utils.image_dataset_from_directory(
    RUTA_TEST, shuffle=False, image_size=TAM_IMG, batch_size=LOTE, class_names=CLASES)
rutas_prueba = test_raw.file_paths          # mismo orden que las predicciones del conjunto de prueba

def optimizar(ds, mezclar=False):
    ds = ds.map(lambda x, y: (tf.cast(x, tf.uint8), y), num_parallel_calls=AUTOTUNE).cache()
    if mezclar:
        ds = ds.shuffle(200, seed=SEMILLA)
    return ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

train_ds = optimizar(train_raw, mezclar=True)
val_ds   = optimizar(val_raw)
test_ds  = optimizar(test_raw)

y_train = np.concatenate([y.numpy() for _, y in train_raw])
y_test  = np.concatenate([y.numpy() for _, y in test_raw])
pesos = compute_class_weight('balanced', classes=np.arange(NUM_CLASES), y=y_train)
PESOS_CLASE = dict(enumerate(pesos))
pd.DataFrame({'imágenes de entrenamiento': np.bincount(y_train), 'peso de la clase': pesos.round(3)}, index=CLASES)

### 5.1 Aumento de datos
El aumento de datos genera variaciones aleatorias de cada imagen **solo durante el entrenamiento** (en predicción estas capas no hacen nada). Esto reduce el sobreajuste y simula condiciones reales de vigilancia: objetos en distintas posiciones, orientaciones, distancias e iluminación.

| Transformación | Parámetro | Motivo |
|---|---|---|
| Volteo horizontal | — | La dirección de vuelo no cambia la clase |
| Rotación | ±8 % de giro (≈ ±29°) | Inclinación de la aeronave |
| Zoom | ±15 % | Distancia al objeto |
| Traslación | ±10 % | Posición del objeto en la imagen |
| Contraste | ±20 % | Condiciones de luz y clima |

In [ ]:
def crear_aumento():
    return keras.Sequential([
        layers.RandomFlip('horizontal'),
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.15),
        layers.RandomTranslation(0.10, 0.10),
        layers.RandomContrast(0.20),
    ], name='aumento_datos')

aumento_demo = crear_aumento()
imagenes, etiquetas = next(iter(train_raw))
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    img = imagenes[0] if i == 0 else aumento_demo(imagenes[:1], training=True)[0]
    plt.imshow(np.clip(img.numpy(), 0, 255).astype('uint8'))
    plt.title('Original' if i == 0 else f'Aumento {i}', fontsize=9); plt.axis('off')
plt.suptitle(f'Aumento de datos sobre una imagen de la clase "{CLASES[etiquetas[0]]}"')
plt.tight_layout(); plt.show()

In [ ]:
def graficar_historial(historiales, titulo, corte=None):
    acc, val_acc, loss, val_loss = [], [], [], []
    for h in historiales:
        acc += h.history['accuracy']; val_acc += h.history['val_accuracy']
        loss += h.history['loss'];    val_loss += h.history['val_loss']
    fig, ejes = plt.subplots(1, 2, figsize=(14, 4))
    ejes[0].plot(acc, label='Entrenamiento'); ejes[0].plot(val_acc, label='Validación')
    ejes[0].set_title('Exactitud'); ejes[0].set_xlabel('Época')
    ejes[1].plot(loss, label='Entrenamiento'); ejes[1].plot(val_loss, label='Validación')
    ejes[1].set_title('Pérdida'); ejes[1].set_xlabel('Época')
    for eje in ejes:
        if corte:
            eje.axvline(corte - 1, color='gray', ls='--', label='Inicio de ajuste fino')
        eje.legend(); eje.grid(alpha=0.3)
    fig.suptitle(titulo); plt.tight_layout(); plt.show()

## 6. Modelo 1: CNN diseñada desde cero
### 6.1 Arquitectura
| Etapa | Capas | Propósito |
|---|---|---|
| Entrada | 224 × 224 × 3 | Imagen RGB |
| Aumento de datos | Volteo, rotación, zoom, traslación, contraste | Generalización |
| Normalización | `Rescaling(1/255)` | Escala los píxeles a [0, 1] |
| 4 bloques convolucionales | 2 × (Conv 3×3 → BatchNorm → ReLU) → MaxPooling; filtros 32, 64, 128, 256 | Extraen bordes, formas y partes del objeto |
| Clasificador | GlobalAveragePooling → Dropout → Dense(256, ReLU) → Dropout → Dense(6, softmax) | Decide la clase |

Se usa regularización L2 en las convoluciones, *Batch Normalization* para estabilizar el entrenamiento y *Dropout* para reducir el sobreajuste.

In [ ]:
def construir_cnn(tasa_aprendizaje=1e-3, dropout=0.4, l2=1e-4, filtros=(32, 64, 128, 256)):
    reg = keras.regularizers.l2(l2)
    entradas = keras.Input(shape=TAM_IMG + (3,))
    x = crear_aumento()(entradas)
    x = layers.Rescaling(1.0 / 255)(x)
    for f in filtros:
        for _ in range(2):
            x = layers.Conv2D(f, 3, padding='same', use_bias=False, kernel_regularizer=reg)(x)
            x = layers.BatchNormalization()(x)
            x = layers.Activation('relu')(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=reg)(x)
    x = layers.Dropout(dropout)(x)
    salidas = layers.Dense(NUM_CLASES, activation='softmax')(x)
    modelo = keras.Model(entradas, salidas, name='cnn_desde_cero')
    modelo.compile(optimizer=keras.optimizers.Adam(tasa_aprendizaje),
                   loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return modelo

construir_cnn().summary()

### 6.2 Búsqueda de hiperparámetros
Se comparan cuatro combinaciones de **tasa de aprendizaje** y **dropout**, entrenando cada una durante pocas épocas. Se elige la que obtiene la mayor **exactitud de validación** (el conjunto de prueba no se usa en esta decisión). Tiempo aproximado: 10 a 15 minutos con GPU; con CPU puede tardar bastante más (puedes cambiar `BUSCAR_HIPERPARAMETROS` a `False`).

In [ ]:
BUSCAR_HIPERPARAMETROS = True     # cambia a False para usar directamente la configuración por defecto
EPOCAS_BUSQUEDA = 8
ESPACIO = [{'tasa_aprendizaje': lr, 'dropout': d} for lr in (1e-3, 3e-4) for d in (0.3, 0.5)]

resultados_busqueda = []
if BUSCAR_HIPERPARAMETROS:
    for cfg in ESPACIO:
        keras.utils.set_random_seed(SEMILLA)
        inicio = time.time()
        h = construir_cnn(**cfg).fit(train_ds, validation_data=val_ds, epochs=EPOCAS_BUSQUEDA,
                                     class_weight=PESOS_CLASE, verbose=0)
        resultados_busqueda.append({**cfg, 'mejor val_accuracy': max(h.history['val_accuracy']),
                                    'val_loss final': h.history['val_loss'][-1],
                                    'minutos': (time.time() - inicio) / 60})
        print(cfg, f"-> val_accuracy = {resultados_busqueda[-1]['mejor val_accuracy']:.4f}")
    busqueda_df = pd.DataFrame(resultados_busqueda).sort_values('mejor val_accuracy', ascending=False)
    MEJOR_CFG = busqueda_df.iloc[0][['tasa_aprendizaje', 'dropout']].to_dict()
    display(busqueda_df.round(4))
else:
    MEJOR_CFG = {'tasa_aprendizaje': 1e-3, 'dropout': 0.4}
print('Hiperparámetros seleccionados:', MEJOR_CFG)

### 6.3 Entrenamiento final
Con los hiperparámetros elegidos se entrena hasta 60 épocas con:
- **EarlyStopping:** detiene el entrenamiento si la pérdida de validación no mejora en 10 épocas y recupera los mejores pesos.
- **ReduceLROnPlateau:** reduce a la mitad la tasa de aprendizaje si la pérdida de validación se estanca 4 épocas.
- **Pesos por clase:** compensan el desbalance de la clase `cohete`.

In [ ]:
keras.utils.set_random_seed(SEMILLA)
modelo1 = construir_cnn(**MEJOR_CFG)
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1),
]
inicio = time.time()
hist1 = modelo1.fit(train_ds, validation_data=val_ds, epochs=60,
                    class_weight=PESOS_CLASE, callbacks=callbacks)
print(f'Tiempo de entrenamiento: {(time.time() - inicio) / 60:.1f} min')
graficar_historial([hist1], 'Modelo 1: CNN desde cero')

In [ ]:
# Exportación del modelo 1
RUTA_M1 = RUTA_MODELOS / 'modelo1_cnn_desde_cero.keras'
modelo1.save(RUTA_M1)
with open(RUTA_MODELOS / 'modelo1_historial.json', 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in hist1.history.items()}, f)
print('Modelo 1 exportado en:', RUTA_M1)

## 7. Modelo 2: modelo pre-entrenado (transferencia de aprendizaje)


In [ ]:
keras.utils.set_random_seed(SEMILLA)
base = keras.applications.MobileNetV2(input_shape=TAM_IMG + (3,), include_top=False, weights='imagenet')
base.trainable = False

entradas = keras.Input(shape=TAM_IMG + (3,))
x = crear_aumento()(entradas)
x = layers.Rescaling(1.0 / 127.5, offset=-1)(x)
x = base(x, training=False)          # BatchNorm de la base siempre en modo inferencia
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
salidas = layers.Dense(NUM_CLASES, activation='softmax')(x)
modelo2 = keras.Model(entradas, salidas, name='mobilenetv2_transferencia')
modelo2.compile(optimizer=keras.optimizers.Adam(1e-3),
                loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelo2.summary()

In [ ]:
# Fase 1: solo se entrena el clasificador
EPOCAS_FASE1 = 15
inicio = time.time()
hist2a = modelo2.fit(train_ds, validation_data=val_ds, epochs=EPOCAS_FASE1, class_weight=PESOS_CLASE,
                     callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=4,
                                                              restore_best_weights=True)])
print(f'Fase 1: {(time.time() - inicio) / 60:.1f} min')

In [ ]:
# Fase 2: ajuste fino de las últimas capas de MobileNetV2
CAPA_INICIO_AJUSTE = 100                     # MobileNetV2 tiene 154 capas; se ajustan de la 100 en adelante
base.trainable = True
for capa in base.layers[:CAPA_INICIO_AJUSTE]:
    capa.trainable = False
print('Capas entrenables de la base:', sum(c.trainable for c in base.layers), 'de', len(base.layers))

modelo2.compile(optimizer=keras.optimizers.Adam(1e-5),
                loss='sparse_categorical_crossentropy', metrics=['accuracy'])
inicio = time.time()
hist2b = modelo2.fit(train_ds, validation_data=val_ds, epochs=15, class_weight=PESOS_CLASE,
                     callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                                              restore_best_weights=True)])
print(f'Fase 2: {(time.time() - inicio) / 60:.1f} min')
graficar_historial([hist2a, hist2b], 'Modelo 2: MobileNetV2 (transferencia de aprendizaje)',
                   corte=len(hist2a.history['loss']) + 1)

In [ ]:
# Exportación del modelo 2
RUTA_M2 = RUTA_MODELOS / 'modelo2_mobilenetv2.keras'
modelo2.save(RUTA_M2)
print('Modelo 2 exportado en:', RUTA_M2)

## 8. Modelo 3: Teachable Machine


In [ ]:
# Descompresión del modelo exportado de Teachable Machine
zip_modelo_tm = RUTA_TM / 'converted_keras.zip'
if zip_modelo_tm.exists():
    with zipfile.ZipFile(zip_modelo_tm) as z:
        z.extractall(RUTA_TM)
print('Contenido de la carpeta del modelo 3:')
for archivo in sorted(RUTA_TM.iterdir()):
    print(f'  {archivo.name:30s} {archivo.stat().st_size / 1e6:8.2f} MB')

In [ ]:
import tf_keras

class DepthwiseConv2DCompatible(tf_keras.layers.DepthwiseConv2D):
    def __init__(self, *args, groups=None, **kwargs):   # 'groups' no existe en esta capa
        super().__init__(*args, **kwargs)

def normalizar_nombre(texto):
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode()
    return texto.strip().lower()

def cargar_modelo_tm():
    ruta_h5 = RUTA_TM / 'keras_model.h5'
    if not ruta_h5.exists():
        print('No se encontró keras_model.h5; completa la sección 8 y vuelve a ejecutar.')
        return None, None
    modelo = tf_keras.models.load_model(ruta_h5, compile=False,
                                        custom_objects={'DepthwiseConv2D': DepthwiseConv2DCompatible})
    # labels.txt tiene líneas como "0 avion"
    lineas = (RUTA_TM / 'labels.txt').read_text(encoding='utf-8').splitlines()
    etiquetas_tm = [normalizar_nombre(l.split(' ', 1)[1]) for l in lineas if l.strip()]
    faltantes = set(CLASES) - set(etiquetas_tm)
    assert not faltantes, f'Las clases de Teachable Machine no coinciden: faltan {faltantes}'
    return modelo, etiquetas_tm

modelo3, etiquetas_tm = cargar_modelo_tm()
if modelo3 is not None:
    print('Etiquetas del modelo de Teachable Machine:', etiquetas_tm)
    modelo3.summary()

## 9. Importación y comparación de los tres modelos


In [ ]:
modelos = {
    'Modelo 1: CNN desde cero': keras.models.load_model(RUTA_M1),
    'Modelo 2: MobileNetV2 (pre-entrenado)': keras.models.load_model(RUTA_M2),
}
predicciones = {}
for nombre, modelo in modelos.items():
    probas = modelo.predict(test_ds, verbose=0)
    predicciones[nombre] = probas.argmax(axis=1)
    print(f'{nombre}: {len(probas):,} imágenes de prueba evaluadas')

def predecir_tm(modelo, rutas, etiquetas_modelo, lote=64):
    # Reordena las salidas de Teachable Machine al orden de CLASES
    orden = [etiquetas_modelo.index(c) for c in CLASES]
    resultado = []
    for i in tqdm(range(0, len(rutas), lote), desc='Teachable Machine'):
        lote_img = []
        for ruta in rutas[i:i + lote]:
            with Image.open(ruta) as img:
                img = ImageOps.fit(img.convert('RGB'), (224, 224), Image.Resampling.LANCZOS)
            lote_img.append(np.asarray(img, dtype=np.float32) / 127.5 - 1)   # mismo preprocesamiento que TM
        resultado.append(modelo.predict(np.stack(lote_img), verbose=0)[:, orden])
    return np.concatenate(resultado).argmax(axis=1)

if modelo3 is not None:
    predicciones['Modelo 3: Teachable Machine'] = predecir_tm(modelo3, rutas_prueba, etiquetas_tm)
else:
    print('Aviso: el modelo 3 no está disponible, la comparación incluye solo los modelos 1 y 2.')

In [ ]:
filas = []
for nombre, y_pred in predicciones.items():
    filas.append({'Modelo': nombre,
                  'Exactitud (accuracy)': accuracy_score(y_test, y_pred),
                  'F1 macro': f1_score(y_test, y_pred, average='macro'),
                  'Aciertos': int((y_pred == y_test).sum()),
                  'Imágenes de prueba': len(y_test)})
comparacion = pd.DataFrame(filas).sort_values('Exactitud (accuracy)', ascending=False).reset_index(drop=True)
comparacion.to_csv(RUTA_PROYECTO / 'comparacion_modelos.csv', index=False)
display(comparacion.style.format({'Exactitud (accuracy)': '{:.2%}', 'F1 macro': '{:.4f}'})
        .highlight_max(subset=['Exactitud (accuracy)'], color='#C8E6C9'))

plt.figure(figsize=(9, 4))
ax = sns.barplot(data=comparacion, y='Modelo', x='Exactitud (accuracy)', color='steelblue')
for i, v in enumerate(comparacion['Exactitud (accuracy)']):
    ax.text(v, i, f' {v:.2%}', va='center')
plt.xlim(0, 1.1); plt.title('Exactitud en el conjunto de prueba'); plt.tight_layout(); plt.show()

In [ ]:
fig, ejes = plt.subplots(1, len(predicciones), figsize=(7 * len(predicciones), 6))
for eje, (nombre, y_pred) in zip(np.atleast_1d(ejes), predicciones.items()):
    mc = confusion_matrix(y_test, y_pred)
    sns.heatmap(mc, annot=True, fmt='d', cmap='Blues', xticklabels=CLASES, yticklabels=CLASES, ax=eje, cbar=False)
    eje.set_title(f'{nombre}\nExactitud = {accuracy_score(y_test, y_pred):.2%}')
    eje.set_xlabel('Predicción'); eje.set_ylabel('Clase real')
plt.tight_layout(); plt.show()

for nombre, y_pred in predicciones.items():
    print('=' * 70, f'\n{nombre}\n', '=' * 70)
    print(classification_report(y_test, y_pred, target_names=CLASES, digits=4))

## 10. Selección del mejor modelo


In [ ]:
MEJOR = comparacion.iloc[0]
print(f"Mejor modelo: {MEJOR['Modelo']}")
print(f"Exactitud en prueba: {MEJOR['Exactitud (accuracy)']:.2%} "
      f"({MEJOR['Aciertos']:,} de {MEJOR['Imágenes de prueba']:,} imágenes)")

# Ejemplos de predicciones del mejor modelo sobre imágenes de prueba
y_mejor = predicciones[MEJOR['Modelo']]
rng = np.random.default_rng(SEMILLA)
indices = rng.choice(len(rutas_prueba), 18, replace=False)
titulos = [f"Real: {CLASES[y_test[i]]}\nPred.: {CLASES[y_mejor[i]]} {'✓' if y_test[i] == y_mejor[i] else '✗'}"
           for i in indices]
mostrar_cuadricula([rutas_prueba[i] for i in indices], titulos,
                   titulo_general=f"Predicciones de {MEJOR['Modelo']}")

## 11. Exportación de los modelos


In [ ]:
nombre_mejor = MEJOR['Modelo']
try:
    modelo_mejor = modelo3 if 'Teachable' in nombre_mejor else modelos[nombre_mejor]
    convertidor = tf.lite.TFLiteConverter.from_keras_model(modelo_mejor)
    ruta_tflite = RUTA_MODELOS / 'mejor_modelo.tflite'
    ruta_tflite.write_bytes(convertidor.convert())
    (RUTA_MODELOS / 'mejor_modelo_clases.txt').write_text('\n'.join(CLASES), encoding='utf-8')
    print('Mejor modelo exportado a TensorFlow Lite:', ruta_tflite)
except Exception as error:
    print('No fue posible convertir a TensorFlow Lite:', error)

print('\nArchivos exportados en resultados/modelos:')
for archivo in sorted(RUTA_MODELOS.rglob('*')):
    if archivo.is_file():
        print(f'  {str(archivo.relative_to(RUTA_MODELOS)):45s} {archivo.stat().st_size / 1e6:8.2f} MB')

## 12. Conclusiones


In [ ]:
print(f'Base de datos: {len(inventario):,} imágenes originales en {NUM_CLASES} clases.')
print(f'Después de la limpieza: {len(limpio):,} imágenes '
      f'({len(inventario) - len(limpio):,} eliminadas por duplicados o etiquetas ambiguas).')
print(f'Entrenamiento: {len(entrenamiento_df):,} imágenes | Prueba: {len(prueba_df):,} imágenes\n')
for _, fila in comparacion.iterrows():
    print(f"  {fila['Modelo']:40s} exactitud = {fila['Exactitud (accuracy)']:.2%}   F1 macro = {fila['F1 macro']:.4f}")
print(f"\nModelo seleccionado para el sistema de seguridad: {MEJOR['Modelo']}")